# BraTS Domain Generalization (MixStyle)

This notebook is a thin driver over the `dg/` package. Each section calls into a library
module instead of inlining code — see `dg/config.py`, `dg/preprocessing.py`, `dg/splits.py`,
`dg/models.py`, `dg/training.py`, and `dg/experiment.py` for the implementation.


## Setup

In [ ]:
import sys, subprocess
packages = [
    ["synapseclient"],
    ["nibabel", "pandas", "openpyxl", "pillow"],
    ["torch", "torchvision", "matplotlib"],
]
for pkgs in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from dg import config, runtime
from dg.preprocessing import (
    download_from_synapse,
    preprocess_zip_to_npz,
    visualize_dataset_samples,
)
from dg.splits import (
    build_loso_site_case_table,
    build_main_split,
    build_site_domain_summary,
    summarize_split,
)
from dg.experiment import (
    DataSplitConfig,
    ModelConfig,
    compare_models,
    prepare_dataloaders_from_config,
    report_experiments,
    run_experiment,
    run_loso_sweep,
)
from dg.training import set_seed

# Optional: override any config defaults here
# config.TOKEN = "<your synapse PAT>"

set_seed(config.EXP_SEED)
print("device =", runtime.DEVICE)


# Dataset

**Backup datasets**
- [OASIS](https://arc.net/l/quote/rwsqdjgl) for training and in-domain test
- [alzheimers-dataset-4-class-of-images](https://www.kaggle.com/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images) for out-domain test
- [ANDI](https://adni.loni.usc.edu/data-samples/adni-data/) (permission required)

**Current** — [BraTS](https://www.synapse.org/Synapse:syn51514105): brain tumor scans from
multiple hospitals, used as domain splits.

**Download instructions**
1. Register and request dataset permission
2. Create a Synapse token for programmatic download
3. Set `config.TOKEN` to your own token before running the download cell


In [ ]:
# Download ZIP (no-op if already present)
zip_path = download_from_synapse(config.ZIP_PATH, token=config.TOKEN)
config.ZIP_PATH = zip_path


## 1. Preprocess ZIP → compact NPZ dataset

Reads the BraTS ZIP, assigns each case to a site, converts 3D volumes to 2D axial
slices, labels each slice tumor/normal, caps slices per case, and writes one
compressed `.npz` per case under `site_xx/`. Produces `index.csv`.


In [ ]:
index_df = preprocess_zip_to_npz()
index_df.head()


In [ ]:
visualize_dataset_samples(config.INDEX_CSV, n_show=8, mode="random")


## 2. Load the index

In [ ]:
index_df = pd.read_csv(Path(config.OUT_ROOT) / "index.csv")

print("\n=== Sites Overview ===")
print(index_df.groupby("site")["case_id"].count().sort_values(ascending=False).head(10))

site_domain_summary = build_site_domain_summary(index_df)
candidate_sites_for_loso = site_domain_summary.loc[
    site_domain_summary["candidate_for_loso"], "site"
].tolist()

print("\n=== Dataset Diagnostics ===")
display(site_domain_summary.sort_values(["candidate_for_loso", "cases"], ascending=[False, False]))
print("Candidate held-out sites for leave-one-site-out:", candidate_sites_for_loso)

display(build_loso_site_case_table(index_df))


# Model

We test [MixStyle](https://arxiv.org/abs/2104.02008) — a feature-statistics mixing
layer that encourages the backbone to learn domain-invariant features. The
implementation lives in `dg/models.py` (`MixStyle`, `ResNet18MixStyle`).


# Experiment

## LOSO Evaluation

Run leave-one-site-out across all candidate sites to find the hardest held-out site
(used as our unseen domain below). The baseline config trains only the FC head.


In [ ]:
loso_cfg = ModelConfig(name="LOSO-FC-Only", train_layers=tuple(), use_mixstyle=False)
loso_results_df = run_loso_sweep(
    loso_cfg,
    sites=candidate_sites_for_loso,
    index_df=index_df,
    epochs=5,
    patience=3,
)
print("\nLOSO results ranked from hardest to easiest held-out site")
display(loso_results_df)


Site 18 was the hardest, so we use it as the out-domain test site.

In [ ]:
TEST_SITES = [18]  # hold out site_18 as unseen domain

split = build_main_split(index_df, test_sites=TEST_SITES)

train_df = split["train_df"]
val_df = split["val_df"]
in_domain_test_df = split["in_domain_test_df"]
out_domain_test_df = split["out_domain_test_df"]

print(
    "Train cases:", train_df["case_id"].nunique(),
    "Val cases:", val_df["case_id"].nunique(),
    "In-domain test cases:", in_domain_test_df["case_id"].nunique(),
    "Out-domain test cases:", out_domain_test_df["case_id"].nunique(),
)
print("\n=== Sites in each split ===")
print("Train sites:", sorted(train_df["site"].unique().tolist()))
print("Val sites:  ", sorted(val_df["site"].unique().tolist()))
print("In-domain test sites:", sorted(in_domain_test_df["site"].unique().tolist()))
print("Out-domain test sites:", sorted(out_domain_test_df["site"].unique().tolist()))

print("\n=== Split Diagnostics ===")
summarize_split("train", train_df)
summarize_split("val", val_df)
summarize_split("in-domain test", in_domain_test_df)
summarize_split("out-domain test", out_domain_test_df)


## MixStyle Variants

In [ ]:
default_data_config = DataSplitConfig(
    name="default",
    train_cases=frozenset(split["train_set"]),
    validation_cases=frozenset(split["val_set"]),
    test_cases=frozenset(split["in_domain_test_set"]),
    out_domain_test_cases=frozenset(split["out_domain_test_set"]),
)

experiments = [
    ModelConfig(name="Baseline-L1", train_layers=(1,), use_mixstyle=False),
    ModelConfig(name="Baseline-L2", train_layers=(2,), use_mixstyle=False),
    ModelConfig(name="Baseline-L3", train_layers=(3,), use_mixstyle=False),
    ModelConfig(name="MixStyle-L1-p0.5-a0.1",  train_layers=(1,), use_mixstyle=True, mixstyle_p=0.5,  mixstyle_a=0.1),
    ModelConfig(name="MixStyle-L2-p0.5-a0.1",  train_layers=(2,), use_mixstyle=True, mixstyle_p=0.5,  mixstyle_a=0.1),
    ModelConfig(name="MixStyle-L3-p0.5-a0.1",  train_layers=(3,), use_mixstyle=True, mixstyle_p=0.5,  mixstyle_a=0.1),
    ModelConfig(name="MixStyle-L2-p0.25-a0.1", train_layers=(2,), use_mixstyle=True, mixstyle_p=0.25, mixstyle_a=0.1),
    ModelConfig(name="MixStyle-L2-p0.75-a0.1", train_layers=(2,), use_mixstyle=True, mixstyle_p=0.75, mixstyle_a=0.1),
    ModelConfig(name="MixStyle-L2-p0.5-a0.3",  train_layers=(2,), use_mixstyle=True, mixstyle_p=0.5,  mixstyle_a=0.3),
    ModelConfig(name="MixStyle-L2-p0.5-a0.5",  train_layers=(2,), use_mixstyle=True, mixstyle_p=0.5,  mixstyle_a=0.5),
]


In [ ]:
results = [run_experiment(cfg, default_data_config, index_df=index_df, epochs=15) for cfg in experiments]
experiment_results_by_name = {r["name"]: r for r in results}


In [ ]:
loaders = prepare_dataloaders_from_config(default_data_config, index_df=index_df)
report_experiments(experiments, loaders, results_by_name=experiment_results_by_name)


## Evaluate models

In [ ]:
loaders = prepare_dataloaders_from_config(default_data_config, index_df=index_df, verbose=False)

compare_models(
    [
        ModelConfig(name="Baseline-L2", train_layers=(2,), use_mixstyle=False),
        ModelConfig(name="MixStyle-L2-p0.75-a0.1", train_layers=(2,), use_mixstyle=True, mixstyle_p=0.75, mixstyle_a=0.1),
    ],
    loaders,
)


In [ ]:
compare_models(
    [
        ModelConfig(name="Baseline-L2", train_layers=(2,), use_mixstyle=False),
        ModelConfig(name="MixStyle-L2-p0.5-a0.5", train_layers=(2,), use_mixstyle=True, mixstyle_p=0.5, mixstyle_a=0.5),
    ],
    loaders,
)


## Most Optimal Variant

In [ ]:
optimal_cfg = ModelConfig(
    name="MixStyle-L3-p0.75-a0.5",
    train_layers=(3,),
    use_mixstyle=True,
    mixstyle_p=0.75,
    mixstyle_a=0.5,
)
optimal = run_experiment(optimal_cfg, default_data_config, index_df=index_df, epochs=15)


In [ ]:
loaders = prepare_dataloaders_from_config(default_data_config, index_df=index_df, verbose=False)
compare_models([optimal_cfg], loaders)


It turns out that it is even worse than `MixStyle-L2-p0.75-a0.1`.